## **Introduction**

I was recently invited to interview for a role on a risk desk, and I thought it would be a good opportunity to revisit one of the most important concepts in market risk: Value at Risk, or VaR.

VaR is one of the most widely used risk measures in finance. Formally, it estimates the maximum expected loss of a portfolio over a given time horizon, for a given confidence level, under normal market conditions. For example, **if a portfolio has a 1-day 95% VaR of $10,000**, this means that, under the model assumptions, **there is a 95% probability that the portfolio will not lose more than $10,000 over one day**. Equivalently, there is a 5% probability that the loss will exceed $10,000.

Intuitively, VaR gives a simple answer to a very practical question: how much money could I lose, over a certain period of time, in a reasonably bad scenario? **It does not tell us the worst loss imaginable**, but it gives a threshold that helps summarize downside risk in a clear and operational way. If a trader, a portfolio manager, or a risk manager wants a quick measure of market exposure, VaR is often one of the first numbers they look at.

This is precisely why VaR is so important. It is used across banks, hedge funds, asset managers, commodity trading firms, and trading desks to **monitor risk, compare exposures across portfolios, define limits, and support capital allocation**. In practice, VaR is a core tool in day-to-day risk management: it helps institutions understand how sensitive their positions are to market moves and whether the level of risk being taken is consistent with their objectives and constraints.

That said, VaR is not computed in a single universal way. There are several approaches, each based on different assumptions about how returns behave. In this notebook, I will cover the three main approaches to VaR:

- Historical VaR
- Parametric VaR
- Monte Carlo VaR

The goal is to understand both the intuition behind the metric and how these three methods can be implemented in Python on a simple portfolio, but also analyze the sensitivities of it when we changes key parameters thanks to plotly interactive graph.

### **Pre-requisite**

*Here we import the needed Python libraries, download prices data and define the common functions needed for the different VaR methods.*

*The data download is done only once at the beginning of the notebook. After that, each VaR method reuses the same market data and portfolio construction steps.*

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm

import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider, IntSlider, SelectMultiple


# ===================================================================
# 1. Retrieve Underlyings Close Prices
# ===================================================================

years = 15
tickers = ['SPY', 'BND', 'GLD', 'QQQ', 'VTI']

### Download Price Data
endDate = dt.datetime.now()
startDate = endDate - dt.timedelta(days=365 * years)

close_df = pd.DataFrame()

for ticker in tickers:
    data = yf.download(ticker, start=startDate, end=endDate)
    close_df[ticker] = data['Close']


# ===================================================================
# 2. Common Functions
# ===================================================================

def compute_log_returns(close_df):
    log_returns = np.log(close_df / close_df.shift(1))
    log_returns = log_returns.dropna()
    return log_returns

def create_equal_weights(tickers):
    weights = np.array([1 / len(tickers)] * len(tickers))
    return weights

def compute_historical_returns(log_returns, weights):
    historical_returns = (log_returns * weights).sum(axis=1)
    return historical_returns

def compute_cov_matrix(log_returns):
    cov_matrix = log_returns.cov() * 252  # Annualized daily covariance matrix
    return cov_matrix

def compute_portfolio_std_dev(weights, cov_matrix):
    portfolio_std_dev = np.sqrt(weights.T @ cov_matrix @ weights)
    return portfolio_std_dev

def compute_portfolio_expected_returns(log_returns, weights):
    portfolio_expected_returns = np.sum(log_returns.mean() * weights)
    return portfolio_expected_returns

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


## **Historical VaR**

The Historical Value at Risk method is the most direct and intuitive way to estimate VaR.
Instead of assuming that returns follow a theoretical distribution, it uses actual **past portfolio returns to measure potential future losses.**

The main idea is simple: if we want to know how much the portfolio could lose in a bad scenario, we can look at the worst outcomes that have already happened in the historical sample. Historical VaR therefore relies entirely on observed market data.

#### **How it is calculated**

1. Compute daily log returns for each securities
2. Build portfolio returns using portfolio weights
3. To move the daily VaR to a multi-day VaR, we sum portfolio returns over a rolling window
4. Compute historical VaR, obtained from the lower tail of the historical return distribtion :

$$ VaR = -Percentile(R^{(d)}, 100 - c *100) * V $$

where :
- $ R^{(d)} $ : the series of rolling portfolio returns over $ d $ days
- $ c $ : the confidence level
- $ V $ : the portfolio value

Minus sign is added because the percentile in the left tail is negative, while VaR is conventionally reported as a positive loss amount


In [ ]:
# ===================================================================
# Compute Historical VaR
# ===================================================================

### 1. Compute daily log returns for each securities
log_returns = compute_log_returns(close_df)

### 2. Build portfolio returns using portfolio weights
portfolio_value = 1000000
weights = create_equal_weights(tickers)
historical_returns = compute_historical_returns(log_returns, weights)

# Define function used in the interactive plot
def compute_historical_var(historical_returns, portfolio_value, confidence_interval=0.95, days=5):

    ### 3. Move the daily VaR to a multi-day VaR
    range_returns = historical_returns.rolling(window=days).sum()
    range_returns = range_returns.dropna()

    ### 4. Compute historical VaR
    VaR = -np.percentile(range_returns, 100 - (confidence_interval * 100)) * portfolio_value

    return range_returns, VaR

confidence_interval = 0.95
days = 5

range_returns, VaR = compute_historical_var(historical_returns,portfolio_value, confidence_interval, days)

# ===================================================================
# Plot Historical VaR into an interactive graph
# ===================================================================
def plot_var(confidence_interval=0.95, days=5):

    range_returns, VaR = compute_historical_var(historical_returns, portfolio_value, confidence_interval, days)
    range_returns_usd = range_returns * portfolio_value  # converting returns to usd losses

    fig = go.Figure()

    fig.add_trace(
        go.Histogram(
            x=range_returns_usd,
            nbinsx=50,
            histnorm='probability density',
            name='Distribution',
            opacity=0.75,
            marker_line_color='white',
            marker_line_width=1
        )
    )

    fig.add_vline(
        x=-VaR,
        line_color='red',
        line_dash='dash',
        line_width=2,
        annotation_text=f"VaR ({confidence_interval:.0%}) = {VaR:,.0f} USD",
        annotation_position="top right"
    )

    fig.update_layout(
        title=f'Distribution of Portfolio {days}-Day Returns (USD Value)',
        xaxis_title=f'{days}-Day Portfolio Return (USD Value)',
        yaxis_title='Frequency',
        bargap=0.05,
        template='plotly_white'
    )

    fig.show()

    print(f"VaR ({confidence_interval:.0%}, {days} jours) : {VaR:,.2f} USD")


interact(
    plot_var,
    confidence_interval=FloatSlider(
        value=0.95,
        min=0.80,
        max=0.999,
        step=0.01,
        description='Confidence'
    ),
    days=IntSlider(
        value=5,
        min=1,
        max=60,
        step=1,
        description='Days'
    )
)

interactive(children=(FloatSlider(value=0.95, description='Confidence', max=0.999, min=0.8, step=0.01), IntSli…

<function __main__.plot_var(confidence_interval=0.95, days=5)>

#### **What we can observe by changing the confidence level and the time horizon**

The interactive chart makes the sensitivity of Historical VaR very easy to see.

**When we increase the confidence level, the VaR becomes larger**. This is intuitive: moving from 90% to 95%, or from 95% to 99%, means that we are looking further into the left tail of the return distribution, that is, at rarer and more adverse outcomes. As a result, the loss threshold increases.

**When we increase the time horizon, the VaR also tends to increase.** A 1-day risk measure captures the loss over a single day, while a 10-day or 30-day VaR aggregates returns over a longer period. The more days we consider, the more room there is for losses to accumulate, which makes the distribution wider and pushes the VaR upward.

#### **Pros & Cons**

**Pros**
- No strong distributional assumption: unlike Parametric VaR, it does not require returns to be normally distributed.
- Simple to implement: once historical returns are available, the calculation is pretty straightforward.
- Naturally captures empirical features: if the historical sample contains volatility clustering, skewness, or fat tails, Historical VaR reflects them automatically.

**Cons**
- Backward-looking: it assumes that the past is informative about the future.. and we know that in practice, its not
- Highly dependent on the sample and may underestimate real risk: results can change significantly depending on the time window chosen. If the dataset does not include enough stressed periods, the VaR can look artificially low.

Overall, Historical VaR is a very good starting point because it is transparent and intuitive, but it should always be interpreted with caution and understood as a model based on past data rather than a guarantee about future losses.

## **Parametric VaR**

The Parametric Value at Risk approach estimates potential losses by assuming that portfolio returns follow a given statistical distribution, most often the normal distribution.

Unlike Historical VaR, which relies directly on the empirical distribution of past returns, Parametric VaR summarizes risk using only a few key parameters: the expected return of the portfolio and its volatility. Under the normality assumption, these two quantities are enough to describe the full distribution of returns

#### **How it is calculated**

1. Compute daily log returns for each securities
2. Build portfolio returns using portfolio weights
3. Estimate portfolio volatility using the covariance matrix of assets return : $ \sigma_p = \sqrt{w^\top \Sigma w} $
4. Convert the portfolio parameters into USD PnL, using :

$$ \mu_{\mathrm{USD}} = \mu_p \times d \times V $$

$$ \sigma_{\mathrm{USD}} = V \times \sigma_p \times \sqrt{\frac{d}{252}} $$

where :
- $ \mu_p $ : the average daily portfolio return
- $ \sigma_p $ : annualized portfolio volatility
- $ V $ : the portflio value

5. Compute Parametric VaR using the quantile of the normal distribution corresponding to the confidence level:

$$ q_{left} = - (\mu_{\mathrm{USD}} + z_{1-c}\sigma_{\mathrm{USD}}) $$

$$ VaR = -q_{left} $$

where :
- $ c $ : the confidence level
- $ z_{1-c} $ : the left-tail quantile of the normal distribution

In [ ]:
# ===================================================================
# 1. Compute Parametric VaR
# ===================================================================

### 1. Compute daily log returns for each securities
log_returns = compute_log_returns(close_df)

### 2. Build portfolio returns using portfolio weights
portfolio_value = 1000000
weights = create_equal_weights(tickers)
historical_returns = compute_historical_returns(log_returns, weights)

### 3. Estimate portfolio volatility using covariance matrix
cov_matrix = compute_cov_matrix(log_returns)
portfolio_std_dev = compute_portfolio_std_dev(weights, cov_matrix)


# Define function used in the interactive plot
def compute_parametric_var(historical_returns, portfolio_std_dev, portfolio_value, confidence_interval=0.95, days=5):

    ### 4. Convert portfolio parameters into USD PnL 
    mu_usd = historical_returns.mean() * days * portfolio_value # USD drift

    sigma_usd = portfolio_value * portfolio_std_dev * np.sqrt(days / 252) # USD Vol

    q_left = mu_usd + norm.ppf(1 - confidence_interval) * sigma_usd # Left-tail quantile of the norm distrib

    VaR = -q_left # VaR shown as a positive loss amount

    return mu_usd, sigma_usd, q_left, VaR


# 4. Compute Parametric VaR
confidence_interval = 0.95
days = 5

mu_usd, sigma_usd, q_left, VaR = compute_parametric_var(historical_returns, portfolio_std_dev, portfolio_value, confidence_interval, days)

# ===================================================================
# Plot the VaR into an interactive plot
# ===================================================================

def plot_parametric_var(confidence_interval=0.95, days=5):

    mu_usd, sigma_usd, q_left, VaR = compute_parametric_var(historical_returns, portfolio_std_dev, portfolio_value, confidence_interval, days)

    # Create range of possible P&L values
    x = np.linspace(mu_usd - 4 * sigma_usd, mu_usd + 4 * sigma_usd, 1000)
    y = norm.pdf(x, loc=mu_usd, scale=sigma_usd)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=x,
            y=y,
            mode='lines',
            name='P&L Distribution'
        )
    )

    x_shaded = x[x <= q_left]
    y_shaded = norm.pdf(x_shaded, loc=mu_usd, scale=sigma_usd)

    fig.add_trace(
        go.Scatter(
            x=np.concatenate([x_shaded, x_shaded[::-1]]),
            y=np.concatenate([y_shaded, np.zeros_like(y_shaded)]),
            fill='toself',
            mode='none',
            name='Left Tail'
        )
    )

    fig.add_vline(
        x=q_left,
        line_color='red',
        line_dash='dash',
        line_width=2,
        annotation_text=f"VaR ({confidence_interval:.0%}) = {VaR:,.0f} USD",
        annotation_position="top right"
    )

    fig.update_layout(
        title=f'Parametric VaR Distribution ({days}-Day Horizon)',
        xaxis_title=f'{days}-Day Portfolio P&L (USD)',
        yaxis_title='Density',
        template='plotly_white'
    )

    fig.show()

    print(f"Parametric VaR at {confidence_interval:.0%} confidence over {days} days: {VaR:,.2f} USD")


interact(
    plot_parametric_var,
    confidence_interval=FloatSlider(
        value=0.95,
        min=0.80,
        max=0.999,
        step=0.01,
        description='Confidence'
    ),
    days=IntSlider(
        value=5,
        min=1,
        max=60,
        step=1,
        description='Days'
    )
)

interactive(children=(FloatSlider(value=0.95, description='Confidence', max=0.999, min=0.8, step=0.01), IntSli…

<function __main__.plot_parametric_var(confidence_interval=0.95, days=5)>

#### **What we can observe by changing the confidence level and the time horizon**

Just like with Historical VaR, when the confidence level increases, the VaR increases as well and when the time horizon increases, the VaR also tends to increase as well for the same reasons as mentionned before.

The difference is that, in the parametric framework, VaR is given by a closed-form formula. This means we can describe the sensitivities mathematically.

Starting from the formula:

$$
\mathrm{VaR} = -\left(\mu_{\mathrm{USD}} + z_{1-c}\sigma_{\mathrm{USD}}\right)
$$

with :

$$
\mu_{\mathrm{USD}} = \mu_p \times d \times V
$$

$$
\sigma_{\mathrm{USD}} = V \times \sigma_p \times \sqrt{\frac{d}{252}}
$$

The full expression becomes :

$$
\mathrm{VaR} = -\left(\mu_p d V + z_{1-c} V \sigma_p \sqrt{\frac{d}{252}}\right)
$$

This formula makes the role of each parameter much easier to interpret :

**Sensitivity to the confidence level**

The confidence level enters the formula above through the normal quantile $ z_{1-c} : $

As the confidence level increases, $ 1 - c $ becomes smaller, which pushes the corresponding quantile further into the left tail of the distribution. This makes VaR larger.

**Sensitivity to the time horizon**

The role of the time horizon appears directly in the formula through $ d $. This shows that the two components do not scale in the same way:
- The drift term scales linearly with $ d $
- the volatility term scales with $\sqrt{d}$

So strictly speaking, VaR is not linear in time

**Sensitivity to volatility**

The sensitivity of Parametric VaR to volatility is given by:

$$
\frac{\partial \mathrm{VaR}}{\partial \sigma_p}
=
- z_{1-c} V \sqrt{\frac{d}{252}}
$$

Since $ z_{1-c} < 0 $ for usual confidence levels, this quantity is positive: higher volatility means higher VaR.

This also shows that VaR is linear in volatility, and that the sensitivity grows with the portfolio value and with the square root of the time horizon.

Intuitively, this makes sense: a more volatile portfolio has a wider distribution of possible outcomes, which increases the size of potential losses in the left tail.

**Sensitivity to drift**

The sensitivity of Parametric VaR to drift is given by:

$$
\frac{\partial \mathrm{VaR}}{\partial \mu_p} = -dV
$$

This means that a higher expected return reduces VaR. It also shows that the relationship is linear in drift, and that the effect becomes larger as the time horizon increases.

Intuitively, if the average return of the portfolio shifts upward, the whole P&L distribution shifts slightly to the right, which reduces the loss threshold. However in practice, this effect is usually much smaller than the volatility effect, especially over short horizons.



#### **Pros & Cons**

**Pros**

- Fast to compute: once mean and volatility are estimated, the VaR calculation is immediate.
- Easy to scale: very practical for large portfolios and regular risk monitoring.
- Uses volatility and correlation explicitly: this makes it closely connected to standard portfolio theory.

**Cons**

- Relies on distributional assumptions: typically assumes that returns are normally distributed (false in practice)
- May underestimate tail risk: extreme market moves are often more frequent than the normal model predicts.
- Sensitive to parameters estimation: if volatility and drift are poorly estimated, VaR will also be inaccurate.

Overall, Parametric VaR is a powerful and efficient risk tool, but its usefulness depends heavily on whether the model assumptions remain reasonable for the portfolio and market environment being studied.

## **Monte-Carlo VaR**

Last but not least, Monte Carlo VaR is often seen as the most flexible VaR framework. Here we simulate a large number of possible future portfolio outcomes and then measure the loss threshold from those simulated scenarios.

We estimate the portfolio’s expected return and volatility from historical data, then simulate many possible portfolio P&L values using random draws from a standard normal distribution. Once this simulated distribution is built, the VaR is obtained by taking the relevant percentile of the left tail.

The simulated portfolio P&L used in the code is:

$$
\mathrm{Scenario\ Return}
=
V \mu_p d
+
V \sigma_p Z \sqrt{\frac{d}{252}}
$$

Where :
- $ Z \sim \mathcal{N}(0,1) $

Basically, the formula says that each simulated portfolio outcome is made of two parts :
- **a drift component**, which represents the average expected return over the horizon
- **a random shock component**, which introduces uncertainty through the simulated normal draw

In a way, this is just the discrete version of the Brownian motion formula applied to portfolio P&L

Once many such scenarios are generated, Monte Carlo VaR is computed as the left-tail percentile of the simulated distribution.

In [ ]:
### 1. Compute daily log returns for each securities
log_returns = compute_log_returns(close_df)

### 2. Build portfolio returns using portfolio weights
portfolio_value = 1000000
weights = create_equal_weights(tickers)
historical_returns = compute_historical_returns(log_returns, weights)

### 3. Estimate portfolio parameters
cov_matrix = compute_cov_matrix(log_returns)
portfolio_std_dev = compute_portfolio_std_dev(weights, cov_matrix)
portfolio_expected_returns = compute_portfolio_expected_returns(log_returns, weights)

# Define function used in the interactive plot
def compute_monte_carlo_var(portfolio_value, portfolio_expected_returns, portfolio_std_dev, confidence_interval=0.95, days=1, simulations=10000):

    z_scores = np.random.normal(0, 1, simulations)

    ### 4. Build simulated P&L Distribution
    scenarioReturn = (portfolio_value * portfolio_expected_returns * days + portfolio_value * portfolio_std_dev * z_scores * np.sqrt(days / 252))

    VaR = -np.percentile(scenarioReturn, 100 * (1 - confidence_interval))

    return scenarioReturn, VaR

### 5. Compute Monte Carlo VaR
confidence_interval = 0.95
scenarioReturn, VaR = compute_monte_carlo_var(portfolio_value, portfolio_expected_returns, portfolio_std_dev, confidence_interval, days, simulations)

# ===================================================================
# Plot the VaR into an interactive plot
# ===================================================================

def plot_monte_carlo_var(confidence_interval=0.95):

    scenarioReturn, VaR = compute_monte_carlo_var(portfolio_value, portfolio_expected_returns, portfolio_std_dev, confidence_interval, days, simulations)

    fig = go.Figure()

    fig.add_trace(
        go.Histogram(
            x=scenarioReturn,
            nbinsx=50,
            histnorm='probability density',
            name='Simulated P&L Distribution',
            opacity=0.75,
            marker_line_color='white',
            marker_line_width=1
        )
    )

    fig.add_vline(
        x=-VaR,
        line_color='red',
        line_dash='dash',
        line_width=2,
        annotation_text=f"VaR ({confidence_interval:.0%}) = {VaR:,.0f} USD",
        annotation_position="top right"
    )

    fig.update_layout(
        title=f'Monte Carlo VaR Distribution ({days}-Day Horizon)',
        xaxis_title=f'{days}-Day Portfolio P&L (USD)',
        yaxis_title='Density',
        bargap=0.05,
        template='plotly_white'
    )

    fig.show()

    print(f"Monte Carlo VaR at {confidence_interval:.0%} confidence over {days} days: {VaR:,.2f} USD")


interact(
    plot_monte_carlo_var,
    confidence_interval=FloatSlider(
        value=0.95,
        min=0.80,
        max=0.999,
        step=0.01,
        description='Confidence'
    )
)

interactive(children=(FloatSlider(value=0.95, description='Confidence', max=0.999, min=0.8, step=0.01), Output…

<function __main__.plot_monte_carlo_var(confidence_interval=0.95)>

In this implementation, the interactive chart only uses the confidence level for simplicity. This works well because changing the confidence level does not change the simulated distribution itself, it only changes the percentile we read from its left tail.

By contrast, changing the time horizon would require generating a new distribution, since both the drift and volatility terms depend on $ d $

#### **What changes compared with the previous methods**

Historical VaR reacts to the empirical return distribution, Parametric VaR reacts through a closed-form formula, while Monte Carlo VaR reacts through the simulated distribution of outcomes. So instead of reading sensitivities directly from past data or from a formula, we observe them through the behavior of the simulation. Overall, sensitivies are roughly the same (since its still the same concept)

**Pros**
- Very flexible: it can be adapted to much richer dynamics than Historical or Parametric VaR.
- Intuitive: it builds a full distribution of possible future P&L outcomes.
- Powerful for complex portfolios: especially useful when payoffs are nonlinear or when simple analytical formulas are not available.

**Cons**
- More computationally intensive: generating many scenarios is slower than using a closed-form formula.
- Model-dependent: results depend on the assumptions used to simulate the scenarios.
- Simulation noise: results can vary slightly from one run to another unless a seed is fixed.
- Can look more sophisticated than it really is: a Monte Carlo result is only as good as the model behind the simulation.

## **Final Remarks**

VaR is useful, but it has an important limitation: it only tells us the loss threshold at a given confidence level. It does not tell us what happens beyond that threshold, in the worst cases. 

That is why people also use Conditional VaR, or Expected Shortfall (ES), which tells us “if we are already in those worst cases, what is the average loss?”, and thus, giving more information about tail risk.

I also made the time horizon interactive in this notebook because it helps build intuition (at least in my case). It is a simple way to see how risk changes when we move from 1 day to 5 days or 10 days.

In practice, however, 1-day risk measures are very common for daily risk monitoring on trading desks, because risk is followed day by day. Regulatory frameworks have also used longer horizons, such as 10 days, for market risk capital.
